# Run HypEx on Spark

[All Quick Tutorials](README.md)

Run A/B, A/A, homogeneity checks, matching, and CUPED on the same data with pandas and Spark. The backend is selected when you create a `Dataset`. Roles, experiment arguments, the `execute(data)` call, and report names are the same in these examples.

## Contents

- [Start Spark](#spark-setup)
- [Choose a Dataset backend](#spark-dataset)
- [Run an A/B test](#spark-ab)
- [Check group homogeneity](#spark-homogeneity)
- [Run an A/A test](#spark-aa)
- [Preserve fixed assignments](#spark-fixed-groups)
- [Match observations](#spark-matching)
- [Apply CUPED](#spark-cuped)
- [Reuse cached Spark data](#spark-persistence)
- [Check backend limitations](#spark-limits)

<a id="spark-setup"></a>

## Start Spark

For local Spark, use Java 17 and set `JAVA_HOME` before starting Jupyter. HypEx installs PySpark 3.5.1. Start a local session with two threads and a temporary checkpoint directory. [Spark 3.5 requirements](https://spark.apache.org/docs/3.5.8/)

`PYSPARK_PYTHON` selects the Python interpreter for Spark workers. The example keeps any existing value.

`master("local[2]")` selects local execution. The UI, progress display, and shuffle partition settings are for this small local run. Set a checkpoint directory in advance for the operations that need it.

In [1]:
from IPython.display import display

import os
import sys
from tempfile import TemporaryDirectory

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

from hypex import AATest, ABTest, HomogeneityTest, Matching
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    FeatureRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import BackendsEnum
from hypex.utils.tutorial_data_creation import gen_control_variates_df

checkpoints = TemporaryDirectory(prefix="hypex-spark-")
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx Spark tutorial")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir(checkpoints.name)
spark.version

'3.5.1'

The output `'3.5.1'` confirms that the local session started. On a cluster, use your existing session and a checkpoint directory accessible to the worker nodes.

<a id="spark-dataset"></a>

## Choose a Dataset backend

The generator returns a pandas DataFrame. Convert it to a Spark DataFrame and pass that object to `Dataset`. If you already loaded a table with `spark.read`, pass it directly.

Create two inputs with the same columns and roles. `backend=BackendsEnum.spark` and `session=spark` apply when creating the Spark input.

HypEx can infer the backend and session from a Spark DataFrame passed as `Dataset(roles=roles, data=spark_frame)`. The example spells them out to show where the backend is chosen. If you pass a pandas DataFrame for execution on Spark, specify `backend=BackendsEnum.spark` and `session=spark`.

In [2]:
def make_inputs(table, roles):
    columns = list(roles)
    return {
        "pandas": Dataset(roles=roles, data=table[columns]),
        "Spark": Dataset(
            roles=roles,
            data=spark.createDataFrame(table[columns]),
            backend=BackendsEnum.spark,
            session=spark,
        ),
    }


frame = gen_control_variates_df(
    data_size=160, dependent_division=False, random_state=42
)
ab_inputs = make_inputs(
    frame,
    {
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "Target_lag_1": PreTargetRole(parent="Target", lag=1),
    },
)
{
    name: {"backend": data.backend_type.value, "columns": data.columns}
    for name, data in ab_inputs.items()
}

{'pandas': {'backend': 'pandas',
  'columns': ['Treatment', 'Target', 'Target_lag_1']},
 'Spark': {'backend': 'spark',
  'columns': ['Treatment', 'Target', 'Target_lag_1']}}

The column lists match, and `backend_type` identifies `pandas` and `spark`. The public `data` property returns the native DataFrame for the backend. You can also move a small existing `Dataset` with `data.to_backend(BackendsEnum.spark, session=spark)`, which preserves the roles.

<a id="spark-ab"></a>

## Run an A/B test

Use `ABTest().execute(data)` on both inputs and display each full `summary`.

Compare `control mean`, `test mean`, `difference`, and `TTest p-value`.

In [3]:
ab = {name: ABTest().execute(data) for name, data in ab_inputs.items()}
for name, result in ab.items():
    print(name)
    display(result.summary)

pandas


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,Target,1,446.976683,450.531864,3.555181,0.795384,-74.128696,81.239057,NOT OK,0.928642


Spark


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,Target,1,446.976683,450.531864,3.555181,0.795384,-74.128696,81.239057,NOT OK,0.928642


The means, estimated difference, and p-value match to the precision shown. Both backends expose `summary` with the same column names.

<a id="spark-homogeneity"></a>

## Check group homogeneity

Keep the assignments in `Treatment` and give the historical metric `TargetRole` so `HomogeneityTest` compares it across groups.

In each full `summary`, compare `difference` and the t-test and KS p-values.

In [4]:
homogeneity_inputs = make_inputs(
    frame, {"Treatment": TreatmentRole(), "Target_lag_1": TargetRole()}
)
homogeneity = {
    name: HomogeneityTest().execute(data)
    for name, data in homogeneity_inputs.items()
}
for name, result in homogeneity.items():
    print(name)
    display(result.summary)

pandas


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,Target_lag_1,1,476.154429,427.073598,-49.080831,-10.307755,-125.507115,27.345452,OK,0.209998,OK,0.064447


Spark


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,Target_lag_1,1,476.154429,427.073598,-49.080831,-10.307755,-125.507115,27.345452,OK,0.209998,OK,0.066307


The difference in means and the t-test p-value match. The KS p-values are close, about 0.064 on pandas and 0.066 on Spark. Spark uses a histogram approximation for this test, which can yield slightly different values through the same API.

<a id="spark-aa"></a>

## Run an A/A test

Pass the historical metric and run two candidate splits per backend with the same `random_states`. Display `best_split_statistics` for both results.

Look for `group`, `difference`, and `TTest p-value`.

In [5]:
aa_inputs = make_inputs(frame, {"Target_lag_1": TargetRole()})
aa = {
    name: AATest(random_states=[10, 11]).execute(data)
    for name, data in aa_inputs.items()
}
for name, result in aa.items():
    print(name)
    display(result.best_split_statistics)

pandas


,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,Target_lag_1,test_1,454.457009,445.479017,-8.977993,-1.975543,OK,0.819296,OK,0.97408


Spark


,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,Target_lag_1,test_1,438.688327,462.05635,23.368023,5.326794,OK,0.553412,OK,0.383162


The reports have the same structure. Estimates differ because the backends use different hashing algorithms for assignment. The same `random_states` makes the split reproducible within a backend.

<a id="spark-fixed-groups"></a>

## Preserve fixed assignments

Label the first ten rows `control` and the next ten `test`. Leave the rest unassigned. `ConstGroupRole` and `AATest.execute` work the same way for both inputs. Collect only two columns from this small sample into pandas to build a compact cross-tabulation.

In [6]:
fixed_frame = frame.assign(pinned_group=None)
fixed_frame.loc[:9, "pinned_group"] = "control"
fixed_frame.loc[10:19, "pinned_group"] = "test"
fixed_inputs = make_inputs(
    fixed_frame,
    {"Target_lag_1": TargetRole(), "pinned_group": ConstGroupRole(str)},
)
checks = {}
for name, data in fixed_inputs.items():
    assigned = AATest(random_states=[42]).execute(data)
    rows = assigned.best_split.select(["pinned_group", "split"]).to_backend(
        BackendsEnum.pandas
    ).data
    checks[name] = pd.crosstab(rows["pinned_group"].fillna("free"), rows["split"])
pd.concat(checks, names=["backend"])

split                 control  test_1
backend pinned_group                 
pandas  control            10       0
        free               70      70
        test                0      10
Spark   control            10       0
        free               64      76
        test                0      10

On both backends, ten fixed users stay in control and ten stay in treatment. The `free` row shows how the remaining users were allocated. The [fixed-assignment guide](../how_to_guide/aa-fixed-groups.ipynb) covers labels and quotas.

<a id="spark-matching"></a>

## Match observations

On a separate small sample, treatment probability depends on `X` and the true treatment effect is 5. Assign `FeatureRole` to `X`, run `Matching` with the same settings on both backends, and compare `summary`.

Mahalanobis is the default distance. `faiss_mode="base"` selects exact search and `quality_tests=[]` disables the balance report. These options configure matching independently of the backend.

In the full reports, find the `ATE` row and the `Effect Size`, `CI Lower`, and `CI Upper` columns.

In [7]:
rng = np.random.default_rng(7)
x = rng.uniform(0, 10, size=120)
treatment = rng.binomial(1, 0.1 + 0.08 * x)
outcome = 20 + 4 * x + 5 * treatment + rng.normal(0, 0.4, size=x.size)
matching_frame = pd.DataFrame(
    {"Treatment": treatment, "Target": outcome, "X": x}
)
matching_inputs = make_inputs(
    matching_frame,
    {"Treatment": TreatmentRole(), "Target": TargetRole(), "X": FeatureRole()},
)
matches = {
    name: Matching(
        distance="mahalanobis",
        faiss_mode="base",
        quality_tests=[],
    ).execute(data)
    for name, data in matching_inputs.items()
}
for name, result in matches.items():
    print(name)
    display(result.summary)

pandas


,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,5.0609,0.1502,0.0,4.7664,5.3554
ATC,5.0142,0.1642,0.0,4.6923,5.3361
ATE,5.0352,0.1328,0.0,4.7749,5.2955


Spark


,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,5.0608,0.1502,0.0,4.7664,5.3552
ATC,5.0142,0.1642,0.0,4.6923,5.3360
ATE,5.0352,0.1328,0.0,4.7749,5.2954


Both backends return ATE, ATT, and ATC. The ATE is about 5.04 in both runs. The [in-depth Matching tutorial](Matching.ipynb) shows how to inspect the matched pairs.

<a id="spark-cuped"></a>

## Apply CUPED

Reuse the A/B inputs and pass the same `cuped_features` mapping. Read `variance_reduction_report` on both backends.

The mapping in `cuped_features` is enough to enable CUPED. The historical role documents the column's meaning, and the mapping determines which column to use.

In [8]:
cuped = {
    name: ABTest(cuped_features={"Target": "Target_lag_1"}).execute(data)
    for name, data in ab_inputs.items()
}
for name, result in cuped.items():
    print(name)
    display(result.variance_reduction_report)

pandas


,feature,variance_reduction_pct
0,Target_cuped,12.30047


Spark


,feature,variance_reduction_pct
0,Target_cuped,12.30047


Both reports show about 12.3% variance reduction. The current CUPED implementation collects two columns on the Spark driver to compute the adjustment coefficient. For large inputs, account for the available driver memory.

<a id="spark-persistence"></a>

## Reuse cached Spark data

If several experiments use the same Spark input, materialize it with `persist(action="count")`. Check the cache before and after the A/B test and again after releasing it.

`storage_level="MEMORY_AND_DISK"` and `action="count"` are the `persist()` defaults. The example spells them out. `blocking=True` waits for Spark to release the cache before checking its state.

In [9]:
spark_input = ab_inputs["Spark"]
spark_input.persist(storage_level="MEMORY_AND_DISK", action="count")
try:
    before = spark_input.is_persisted
    cached = ABTest().execute(spark_input)
    after = spark_input.is_persisted
finally:
    spark_input.unpersist(blocking=True)
{
    "cached before A/B": before,
    "cached after A/B": after,
    "cached after unpersist": spark_input.is_persisted,
}

{'cached before A/B': True,
 'cached after A/B': True,
 'cached after unpersist': False}

The values `True`, `True`, and `False` show the manually managed cache lifecycle. HypEx releases the caches it creates for an experiment. You can reuse a cache you created and release it when you finish.

<a id="spark-limits"></a>

## Check backend limitations

The examples share the HypEx API. Support for some options depends on the backend. For `multitest_method="quantile"` or end-to-end CUPAC with the current models, use pandas. See the [pandas CUPAC example](CUPED_CUPAC.ipynb#cupac).

Use `data.iloc[:5]` to preview five rows through the same `Dataset` interface on either backend. Stop the local session and remove its temporary directory after the examples.

In [10]:
spark.stop()
checkpoints.cleanup()
"Spark session stopped"

'Spark session stopped'

The message confirms that the cleanup cell ran. Start a new session with the setup cell to repeat the examples.